In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
# %%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_orders, load_customer, q18
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [5]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS) 

In [6]:

### cell 2 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [7]:
# %%time
# ### cell 3 ###

# # Optimized cell 3
# gb1 = lineitem.groupby("L_ORDERKEY", as_index=False, sort=False)["L_QUANTITY"].sum()
# # filter orders with total quantity > 300
# gb1 = gb1.loc[gb1.L_QUANTITY > 300]
# # project only needed columns from orders and customer to reduce data movement
# orders_sub   = orders[["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_TOTALPRICE"]]
# customer_sub = customer[["C_CUSTKEY", "C_NAME"]]
# # join and sort, avoiding a second groupby
# total = (
#     gb1
#       .merge(orders_sub,   left_on="L_ORDERKEY", right_on="O_ORDERKEY")
#       .merge(customer_sub, left_on="O_CUSTKEY", right_on="C_CUSTKEY")
#       [["C_NAME", "C_CUSTKEY", "O_ORDERKEY", "O_ORDERDATE", "O_TOTALPRICE", "L_QUANTITY"]]
#       .sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])
# )

In [8]:
# total

In [9]:
%%time

gb1 = lineitem.groupby("L_ORDERKEY", as_index=False, sort=False)["L_QUANTITY"].sum()
fgb1 = gb1[gb1.L_QUANTITY > 300]
jn1 = fgb1.merge(orders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")
jn2 = jn1.merge(customer, left_on="O_CUSTKEY", right_on="C_CUSTKEY")
gb2 = jn2.groupby(
    ["C_NAME", "C_CUSTKEY", "O_ORDERKEY", "O_ORDERDATE", "O_TOTALPRICE"],
    as_index=False,
    sort=False,
)["L_QUANTITY"].sum()
total = gb2.sort_values(["O_TOTALPRICE", "O_ORDERDATE"], ascending=[False, True])


CPU times: user 293 ms, sys: 119 ms, total: 412 ms
Wall time: 394 ms


In [9]:
total

,C_NAME,C_CUSTKEY,O_ORDERKEY,O_ORDERDATE,O_TOTALPRICE,L_QUANTITY
440,Customer#001287812,1287812,42290181,1997-11-26,558289.17,318
23,Customer#000644812,644812,2745894,1996-07-04,557664.53,304
380,Customer#001172513,1172513,36667107,1997-06-06,550142.18,322
456,Customer#000399481,399481,43906817,1995-04-06,549431.65,312
219,Customer#000571654,571654,21213895,1992-01-03,549380.08,327
...,...,...,...,...,...,...
328,Customer#001263409,1263409,31855814,1998-04-10,365844.60,309
494,Customer#001470748,1470748,47452993,1993-01-06,365823.23,301
125,Customer#000248779,248779,12563331,1993-06-02,365315.83,301
274,Customer#000653965,653965,27458500,1995-09-25,364945.75,304


In [ ]:
### cell 4 ###

total.info()